# UACB-604 · Lecture 1.3 — How genetic information is organised

A genome sequence is only half the data. The other half is the **annotation**: where the genes are, which strand they are on, where their exons start and stop.
GenBank files carry both. Today we read annotations with **Biopython** and measure how a bacterial and a eukaryotic genome are organised.

**You will:**
1. Parse a GenBank file and list its features
2. Measure gene density and the fraction of the genome that codes for protein
3. Test whether genes prefer the leading strand of replication
4. Find the signature of **operons** in the distances between neighbouring genes
5. Compare with a yeast chromosome, which has introns

Run each cell with **Shift + Enter**. Cells marked ✏️ are exercises.

In [ ]:
!pip -q install biopython

In [ ]:
import io, urllib.request, collections
import numpy as np
import matplotlib.pyplot as plt
from Bio import SeqIO

def fetch_genbank(acc):
    url = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
           f"?db=nuccore&id={acc}&rettype=gbwithparts&retmode=text")
    text = urllib.request.urlopen(url).read().decode()
    return SeqIO.read(io.StringIO(text), "genbank")

ecoli = fetch_genbank("NC_000913.3")
print(ecoli.description)
print(f"length = {len(ecoli.seq):,} bp, features = {len(ecoli.features):,}")
print(collections.Counter(f.type for f in ecoli.features).most_common(10))

## 1. What does one annotated gene look like?
Each feature has a **type** (gene, CDS, tRNA, rRNA…), a **location** (start, end, strand) and **qualifiers** (name, product, protein translation…).
Note that Biopython uses **0-based, half-open** coordinates: a feature shown as `[336:2799](+)` covers positions 337–2799 in the 1-based numbering you see on NCBI. (Much more on coordinate systems in Lecture 1.5.)

In [ ]:
cds = [f for f in ecoli.features if f.type == "CDS"]
f = next(c for c in cds if c.qualifiers.get("gene", [""])[0] == "lacZ")
print(f.location)
for k in ("gene", "product", "locus_tag"):
    print(f"{k:10s}", f.qualifiers.get(k, ["-"])[0])
print("length (bp):", len(f.location), " → codons:", len(f.location) // 3)

## 2. Gene density and coding fraction
Genes can overlap, so we can't just add up their lengths. We mark every base covered by at least one CDS and count the marked bases (the **union** of intervals).

In [ ]:
def coding_fraction(record, feature_type="CDS"):
    covered = np.zeros(len(record.seq), dtype=bool)
    n = 0
    for f in record.features:
        if f.type == feature_type:
            n += 1
            for part in f.location.parts:          # parts = exons for spliced genes
                covered[int(part.start):int(part.end)] = True
    return n, covered.mean()

n_cds, frac = coding_fraction(ecoli)
mb = len(ecoli.seq) / 1e6
print(f"CDS features: {n_cds:,}")
print(f"genes per Mb: {n_cds / mb:,.0f}")
print(f"coding fraction: {frac:.1%}")

lengths = np.array([len(f.location) for f in cds])
plt.hist(lengths, bins=80, color="slateblue")
plt.axvline(np.median(lengths), color="red", label=f"median {np.median(lengths):.0f} bp")
plt.axvline(lengths.mean(), color="orange", label=f"mean {lengths.mean():.0f} bp")
plt.xlabel("CDS length (bp)"); plt.ylabel("number of genes"); plt.legend(); plt.show()

✏️ **Exercise 1.** Why is the mean larger than the median? Which single gene is the longest, and what does it encode?

In [ ]:
# your code here

## 3. Do genes prefer the leading strand?
DNA replication starts at *oriC* (~3.93 Mb, found with GC skew in Lecture 1.1) and runs both ways round the circle to the terminus (~1.6 Mb).
On the half that runs clockwise from *oriC*, the **leading strand** is the + strand; on the other half it is the − strand.
If gene orientation were random, about 50% of genes would be on the leading strand.

In [ ]:
ORI, TER, L = 3_925_000, 1_600_000, len(ecoli.seq)

def clockwise_from_ori(pos):
    # True if pos lies on the replichore that runs clockwise (increasing coordinate) from ori to ter
    return (pos - ORI) % L < (TER - ORI) % L

leading = 0
for f in cds:
    mid = (int(f.location.start) + int(f.location.end)) // 2
    on_plus = f.location.strand == 1
    leading += on_plus if clockwise_from_ori(mid) else not on_plus
p_hat = leading / len(cds)
se = np.sqrt(0.25 / len(cds))
print(f"genes on the leading strand: {p_hat:.1%}  (expected 50% ± {1.96*se:.1%} if random)")

✏️ **Exercise 2.** Is the bias statistically significant? Compute a z-score $z = (\hat p - 0.5)/\sqrt{0.25/n}$. One explanation: if a gene is on the lagging strand, the replication machinery collides head-on with RNA polymerase. Is that a mechanism or just a story? How would you test it?

In [ ]:
# your code here

## 4. Operons leave a signature in the gaps between genes
In an operon, neighbouring genes sit on the **same strand** and are often packed tightly, sometimes even **overlapping** by a few bases (e.g. `ATGA`: the `TGA` stop of one gene overlaps the `ATG` start of the next).
Neighbours on **opposite strands** can't share an operon.

So: compare the gap (next start − previous end) for same-strand pairs vs opposite-strand pairs.

In [ ]:
genes = sorted(cds, key=lambda f: int(f.location.start))
same, opp = [], []
for a, b in zip(genes, genes[1:]):
    gap = int(b.location.start) - int(a.location.end)
    if gap < -100 or gap > 600:
        continue                     # skip odd cases and long gaps for the plot
    (same if a.location.strand == b.location.strand else opp).append(gap)

print(f"adjacent pairs on the same strand: {len(same)/(len(same)+len(opp)):.1%}")
bins = np.arange(-60, 400, 10)
plt.hist(same, bins=bins, alpha=0.6, label="same strand")
plt.hist(opp, bins=bins, alpha=0.6, label="opposite strands")
plt.xlabel("gap between neighbouring genes (bp, negative = overlap)"); plt.ylabel("pairs"); plt.legend(); plt.show()
gaps = collections.Counter(same)
print("most common same-strand gaps:", gaps.most_common(5))

✏️ **Exercise 3.**
1. Which gap values are the most common for same-strand pairs? Explain −4 and −1 using the stop codons TAA, TAG, TGA and the start codon ATG.
2. Propose a simple rule ("same strand and gap < X bp") for predicting that two genes are in one operon. How would you choose X? (This is how early operon predictors worked.)

In [ ]:
# your code here

## 5. A eukaryote: yeast chromosome III
*Saccharomyces cerevisiae* is a compact eukaryote. Some of its genes have introns, so a CDS location is a **join** of several parts (exons).

In [ ]:
yeast = fetch_genbank("NC_001135.5")
print(yeast.description, f"{len(yeast.seq):,} bp")
ycds = [f for f in yeast.features if f.type == "CDS"]
n_y, frac_y = coding_fraction(yeast)
spliced = [f for f in ycds if len(f.location.parts) > 1]
print(f"CDS: {n_y}, genes per Mb: {n_y / (len(yeast.seq)/1e6):.0f}, coding fraction: {frac_y:.1%}")
print(f"CDS with introns: {len(spliced)}")
for f in spliced:
    print(" ", f.qualifiers.get("locus_tag", ["?"])[0], f.qualifiers.get("gene", [""])[0], f.location)

✏️ **Exercise 4.** Fill in the table and discuss: what changes from bacterium → yeast → human (use the lecture's human numbers)?

| | *E. coli* | yeast chr III | human |
|---|---|---|---|
| genes per Mb | | | ~6.5 |
| coding fraction | | | ~1–2% |
| genes with introns | | | almost all |

---
**Further reading:** Brown, *Genomes 5*, ch. 7 (eukaryotic nuclear genomes) and ch. 8 (prokaryotic and organelle genomes) · Pevsner, ch. 17 (bacteria and archaea) and ch. 18 (fungi).